# RL Training — CEFR Text Simplification (Russian)

**Model:** `bond005/FRED-T5-large-instruct-v0.1`  
**Algorithm:** PPO via TRL 0.9.4  
**Structure:**
1. `CELL 0` — Install dependencies
2. `CELL 1` — Imports, helpers, checkpoint functions, `run_training`
3. `CELL 2` — Configuration (only cell you edit)
4. `CELL 3` — Run training
5. `CELL 4` — Smoke test / mini debug (optional, run before Cell 3 to verify pipeline)

In [ ]:
#  CELL 0 — Install dependencies

!pip install -q trl==0.9.4
!pip install -q transformers==4.40.0
!pip install -q peft==0.10.0
!pip install -q bitsandbytes
!pip install -q pymorphy3
!pip install -q safetensors

In [ ]:
# rerun the environment
!pip uninstall -y bitsandbytes
!pip install -U --no-cache-dir bitsandbytes==0.46.0

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import torch
from transformers import AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer, create_reference_model
from typing import List, Optional

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
vocabulary = pd.read_csv("/content/drive/MyDrive/ru_text_simplification_1/vocabulary.csv", sep=';', encoding='cp1251')
vocabulary.rename(columns={'word': 'base'}, inplace=True)
vocabulary.to_csv('new_vocabulary.csv', index=False)
vocabulary

In [ ]:
# Config

from dataclasses import dataclass
from typing import Optional, List, Literal


@dataclass
class PairwiseRewardModelConfig:
    model_name: str = "openai-community/gpt2"
    num_labels: int = 1
    level: str = "A" # "A", "B", or "C"
    train_batch_size: int = 32
    eval_batch_size: int = 32
    learning_rate: float = 1.41e-5
    # changed to 5
    num_train_epochs: int = 5
    gradient_accumulation_steps: int = 2
    output_dir: str = "./reward_model/"
    max_length: int = 128
    wandb_project: str = "CEFR-RewardModel"
    wandb_exp_name: Optional[str] = None

@dataclass
class RLTrainingConfig:
    training_sentences_path: Optional[str] = None
    # CHANGED: default model
    model_name: str = "bond005/FRED-T5-large-instruct-v0.1"
    reward_model_path: str = "./reward_model/"
    level: str = "A"
    batch_size: int = 2
    mini_batch_size: int = 1
    generate_bs: int = 2
    learning_rate: float = 1e-4
    ppo_epochs: int = 2
    max_new_tokens: int = 64
    gradient_accumulation_steps: int = 2
    output_dir: str = "./rl_model/"
    wandb_project: str = "PPO-training-sentence-simplification"
    wandb_exp_name: Optional[str] = None
    word_list_path: str = "./new_vocabulary.csv"
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_target_modules: Optional[List[str]] = None
    reward_max_length: int = 128

    def __post_init__(self):
        if self.level not in ("A", "B", "C"):
            raise ValueError(f"level must be A, B, or C, got {self.level!r}")
        if self.batch_size < 2:
            import warnings
            warnings.warn("batch_size=1 disables PPO reward normalization (std undefined)")

@dataclass
class BeamSearchConfig:
    base_model_path: str
    condition_model_path: str
    condition_tokenizer_path: str
    level: int  # CEFR level 1-6
    condition_type: Literal["bert", "gpt2"]
    condition_lambda: float = 0.8
    topk: int = 200
    num_beams: int = 5
    max_new_tokens: int = 100
    do_sample: bool = False

In [ ]:
#  CELL 1 — Imports, helpers, checkpoint functions, run_training

import json
import shutil
import warnings
import torch
import nltk
import trl
import pkg_resources
from tqdm.auto import tqdm
from collections import deque
from data_utils import read_complicated_lines

nltk.download('punkt',     quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)

import logging
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("trl").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning)

if pkg_resources.parse_version(trl.__version__) > pkg_resources.parse_version("0.10.1"):
    warnings.warn(f"TRL {trl.__version__} not fully tested — expected <= 0.10.1")
print(f"TRL version: {trl.__version__}")


# Checkpoint helpers
CHECKPOINT_META_FILE = "checkpoint_meta.json"


def save_checkpoint(trainer, output_dir, step, epoch, gdrive_dir=None):
    ckpt_path = os.path.join(output_dir, f"checkpoint_epoch{epoch}_step{step}")
    os.makedirs(ckpt_path, exist_ok=True)
    trainer.ppo_trainer.save_pretrained(ckpt_path)
    meta = {"epoch": epoch, "step": step, "model_name": trainer.config.model_name}
    with open(os.path.join(ckpt_path, CHECKPOINT_META_FILE), "w") as f:
        json.dump(meta, f)
    latest_path = os.path.join(output_dir, "latest_checkpoint")
    if os.path.islink(latest_path):
        os.remove(latest_path)
    elif os.path.exists(latest_path):
        shutil.rmtree(latest_path)
    shutil.copytree(ckpt_path, latest_path)
    tqdm.write(f"Checkpoint saved: {ckpt_path}")
    if gdrive_dir:
        gdrive_ckpt   = os.path.join(gdrive_dir, f"checkpoint_epoch{epoch}_step{step}")
        gdrive_latest = os.path.join(gdrive_dir, "latest_checkpoint")
        try:
            if os.path.exists(gdrive_ckpt):
                shutil.rmtree(gdrive_ckpt)
            shutil.copytree(ckpt_path, gdrive_ckpt)
            if os.path.exists(gdrive_latest):
                shutil.rmtree(gdrive_latest)
            shutil.copytree(ckpt_path, gdrive_latest)
            tqdm.write(f"Mirrored to Drive: {gdrive_ckpt}")
        except Exception as e:
            tqdm.write(f"[WARNING] Drive mirror failed (training continues): {e}")


def find_latest_checkpoint(output_dir, gdrive_dir=None):
    for search_dir in [d for d in [output_dir, gdrive_dir] if d]:
        latest    = os.path.join(search_dir, "latest_checkpoint")
        meta_file = os.path.join(latest, CHECKPOINT_META_FILE)
        if os.path.exists(meta_file):
            with open(meta_file) as f:
                meta = json.load(f)
            meta["path"] = latest
            print(f"Found checkpoint: epoch {meta['epoch']}, step {meta['step']}")
            return meta
    return None


def load_checkpoint_into_trainer(trainer, ckpt_path):
    from peft import set_peft_model_state_dict
    from safetensors.torch import load_file as safetensors_load
    adapter_file = os.path.join(ckpt_path, "adapter_model.safetensors")
    bin_file     = os.path.join(ckpt_path, "adapter_model.bin")
    if os.path.exists(adapter_file):
        state_dict = safetensors_load(adapter_file)
    elif os.path.exists(bin_file):
        state_dict = torch.load(bin_file, map_location="cpu")
    else:
        raise FileNotFoundError(f"No adapter weights found in {ckpt_path}")
    set_peft_model_state_dict(trainer.model.pretrained_model, state_dict)
    print(f"Weights loaded from: {ckpt_path}")


# Response validation helper
# Filters all-padding tensors before ppo_trainer.step()
# to prevent IndexError: mask.nonzero()[-1]
def _is_valid_response(r, pad_token_id):
    if r.numel() == 0:
        return False
    r_flat = r.squeeze()
    if r_flat.dim() == 0:
        return False
    return (r_flat != pad_token_id).sum().item() > 0


# Main training loop
def run_training(rl_trainer, config, all_prompts,
                 start_epoch, start_step, generation_kwargs,
                 output_dir, gdrive_dir, checkpoint_every):

    total_batches = len(all_prompts)
    total_steps   = config.ppo_epochs * total_batches

    print(f"\n{'='*55}")
    print(f"  Sentences    : {len(all_prompts) * config.batch_size}")
    print(f"  Batches/epoch: {total_batches}")
    print(f"  Epochs       : {config.ppo_epochs}")
    print(f"  Total steps  : {total_steps}")
    print(f"  Checkpoints  : every {checkpoint_every} steps")
    print(f"  Saving to    : {output_dir}")
    print(f"{'='*55}\n")

    reward_history = deque(maxlen=50)
    kl_history     = deque(maxlen=50)
    policy_history = deque(maxlen=50)

    global_bar = tqdm(
        total         = total_steps,
        initial       = start_epoch * total_batches + start_step,
        desc          = "PPO training",
        unit          = "step",
        dynamic_ncols = True,
        colour        = "green",
    )

    epoch_bar = None

    try:
        for epoch in range(start_epoch, config.ppo_epochs):
            generation_in_epoch = []

            epoch_bar = tqdm(
                total         = total_batches,
                initial       = start_step if epoch == start_epoch else 0,
                desc          = f"Epoch {epoch+1}/{config.ppo_epochs}",
                unit          = "batch",
                dynamic_ncols = True,
                leave         = False,
                colour        = "blue",
            )

            for idx, batch_prompts in enumerate(all_prompts):

                if epoch == start_epoch and idx <= start_step:
                    continue

                try:
                    # encode inputs
                    batch_inputs_tensors = []
                    for prompt_text in batch_prompts:
                        enc = rl_trainer.tokenizer(
                            prompt_text,
                            return_tensors = "pt",
                            truncation     = True,
                            max_length     = config.reward_max_length,
                            padding        = False,
                        )
                        batch_inputs_tensors.append(enc["input_ids"][0].to("cuda"))

                    # generate
                    with torch.no_grad():
                        response = rl_trainer.ppo_trainer.generate(
                            batch_inputs_tensors,
                            batch_size     = config.generate_bs,
                            return_prompt  = False,
                            remove_padding = True,
                            **generation_kwargs,
                        )

                    # filter all-padding responses before ppo_trainer.step()
                    pad_id = rl_trainer.tokenizer.pad_token_id
                    valid_pairs = [
                        (inp, r)
                        for inp, r in zip(batch_inputs_tensors, response)
                        if _is_valid_response(r, pad_id)
                    ]

                    if not valid_pairs:
                        tqdm.write(f"[SKIP] batch {idx} — all responses are padding")
                        del batch_inputs_tensors, response
                        epoch_bar.update(1)
                        global_bar.update(1)
                        continue

                    batch_inputs_tensors, response = zip(*valid_pairs)
                    batch_inputs_tensors = list(batch_inputs_tensors)
                    response             = list(response)

                    # decode
                    response_text_raw = [
                        rl_trainer.tokenizer.decode(
                            r.squeeze(), skip_special_tokens=True
                        )
                        for r in response
                    ]
                    response_text = [
                        rl_trainer._clean_generated_text(t)
                        for t in response_text_raw
                    ]

                    if all(t.strip() == "" for t in response_text):
                        tqdm.write(f"[SKIP] batch {idx} — empty text after decoding")
                        del batch_inputs_tensors, response
                        epoch_bar.update(1)
                        global_bar.update(1)
                        continue

                    generation_in_epoch.extend(response_text)

                    # compute rewards
                    score = [
                        torch.tensor(0.3 * max(-2.0, min(2.0,
                            rl_trainer.lexical_reward_model.compute_reward(t))))
                        for t in response_text
                    ]
                    reward_model_score_pt = rl_trainer._compute_pairwise_rewards(response_text)
                    reward_score          = [0.7 * r for r in reward_model_score_pt]

                    batch_score = []
                    for clean_text, s1, s2 in zip(response_text, score, reward_score):
                        penalty = torch.tensor(rl_trainer._format_penalty(clean_text))
                        batch_score.append(s1 + s2 + penalty)

                    # PPO step
                    stats = rl_trainer.ppo_trainer.step(
                        batch_inputs_tensors, response, batch_score
                    )

                    # metrics
                    mean_r = rl_trainer._to_float(stats.get("ppo/mean_scores", 0))
                    kl     = rl_trainer._to_float(stats.get("objective/kl",    0))
                    p_loss = rl_trainer._to_float(stats.get("ppo/loss/policy", 0))

                    reward_history.append(mean_r)
                    kl_history.append(kl)
                    policy_history.append(p_loss)

                    avg_r  = sum(reward_history) / len(reward_history)
                    avg_kl = sum(kl_history)      / len(kl_history)

                    # early stopping: KL divergence
                    if len(kl_history) == 50 and avg_kl > 8.0:
                        tqdm.write(
                            f"\n[EARLY STOP] KL too high: {avg_kl:.3f} "
                            f"at epoch {epoch+1} step {idx}"
                        )
                        save_checkpoint(rl_trainer, output_dir, idx, epoch, gdrive_dir)
                        raise StopIteration("KL threshold exceeded")

                    # early stopping: reward regression
                    if len(reward_history) == 50 and epoch > 0:
                        first_half  = list(reward_history)[:25]
                        second_half = list(reward_history)[25:]
                        if sum(first_half) / 25 > sum(second_half) / 25 + 0.05:
                            tqdm.write(
                                f"\n[EARLY STOP] Reward declining at "
                                f"epoch {epoch+1} step {idx}"
                            )
                            save_checkpoint(rl_trainer, output_dir, idx, epoch, gdrive_dir)
                            raise StopIteration("Reward regression detected")

                    # update progress bars
                    postfix = {
                        "r"    : f"{mean_r:.3f}",
                        "r_50" : f"{avg_r:.3f}",
                        "kl"   : f"{kl:.3f}",
                        "kl_50": f"{avg_kl:.3f}",
                        "loss" : f"{p_loss:.2e}",
                    }
                    epoch_bar.set_postfix(postfix)
                    global_bar.set_postfix(postfix)
                    epoch_bar.update(1)
                    global_bar.update(1)

                    # one sample per epoch at midpoint
                    if idx == total_batches // 2 and response_text:
                        tqdm.write(f"\n[epoch {epoch+1} midpoint sample]")
                        tqdm.write(f"  -> {response_text[0][:120]}")

                    # periodic checkpoint
                    if idx > 0 and idx % checkpoint_every == 0:
                        tqdm.write(f"\nCheckpoint — epoch {epoch+1}, step {idx}")
                        save_checkpoint(rl_trainer, output_dir, idx, epoch, gdrive_dir)

                    del batch_inputs_tensors, response, reward_model_score_pt, \
                        batch_score, stats

                except StopIteration:
                    raise

                except KeyboardInterrupt:
                    tqdm.write("\nInterrupted — saving emergency checkpoint...")
                    save_checkpoint(rl_trainer, output_dir, idx, epoch, gdrive_dir)
                    raise

                except Exception as e:
                    tqdm.write(f"[ERROR] epoch {epoch+1} batch {idx}: {e}")
                    torch.cuda.empty_cache()

            epoch_bar.close()

            if generation_in_epoch:
                tqdm.write(f"\n[epoch {epoch+1} final sample]")
                tqdm.write(f"  -> {generation_in_epoch[-1][:120]}")

            tqdm.write(f"\nEpoch {epoch+1} done — updating lexical rewards...")
            rl_trainer.lexical_reward_model.update_rewards(generation_in_epoch)
            save_checkpoint(rl_trainer, output_dir, total_batches, epoch, gdrive_dir)

    except StopIteration as e:
        tqdm.write(f"\nTraining stopped early: {e}")

    finally:
        if epoch_bar is not None:
            epoch_bar.close()
        global_bar.close()


print("Cell 1 loaded — proceed to Cell 2")

In [ ]:
#  CELL 2 — Configuration

MODEL_NAME        = "bond005/FRED-T5-large-instruct-v0.1"
CEFR_LEVEL        = "A"          # "A", "B", or "C"

REWARD_MODEL_PATH = "/content/reward_model_A/content/reward_model"
TRAIN_DATA_PATH   = "/content/drive/MyDrive/ru_text_simplification_1/train_source_target_sentences_cleaned.csv"
WORD_LIST_PATH    = "new_vocabulary.csv"

OUTPUT_DIR        = "/content/drive/MyDrive/ru_text_simplification_1/rl_model_fredT5_A"
GDRIVE_DIR        = "/content/drive/MyDrive/ru_text_simplification_1/rl_checkpoints"

PPO_EPOCHS        = 3      # recommended: 3 for 1k-5k sentences on T4
BATCH_SIZE        = 4      # minimum 2
LEARNING_RATE     = 5e-5   # conservative for short runs
MAX_NEW_TOKENS    = 64
CHECKPOINT_EVERY  = 50     # save to Drive every N steps
MAX_SENTENCES     = 200    # limit for quick runs; set to None for full training

LORA_R            = 16
LORA_ALPHA        = 32
LORA_DROPOUT      = 0.05

# Resume after disconnect
RESUME            = False

print("=" * 50)
print(f"  Model      : {MODEL_NAME}")
print(f"  Level      : {CEFR_LEVEL}")
print(f"  Epochs     : {PPO_EPOCHS}")
print(f"  Batch size : {BATCH_SIZE}")
print(f"  LR         : {LEARNING_RATE}")
print(f"  Sentences  : {MAX_SENTENCES if MAX_SENTENCES else 'all'}")
print(f"  Resume     : {RESUME}")
print(f"  Output     : {OUTPUT_DIR}")
print("=" * 50)
print("Config ready — run Cell 3 to start training")

In [ ]:
#  CELL 3 — Run training

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(GDRIVE_DIR, exist_ok=True)

config = RLTrainingConfig(
    model_name                  = MODEL_NAME,
    reward_model_path           = REWARD_MODEL_PATH,
    level                       = CEFR_LEVEL,
    word_list_path              = WORD_LIST_PATH,
    training_sentences_path     = TRAIN_DATA_PATH,
    output_dir                  = OUTPUT_DIR,
    wandb_project               = "align-simplification-test",
    wandb_exp_name              = "PPO",
    batch_size                  = BATCH_SIZE,
    mini_batch_size             = 1,
    generate_bs                 = BATCH_SIZE,
    learning_rate               = LEARNING_RATE,
    ppo_epochs                  = PPO_EPOCHS,
    max_new_tokens              = MAX_NEW_TOKENS,
    gradient_accumulation_steps = 2,
    lora_r                      = LORA_R,
    lora_alpha                  = LORA_ALPHA,
    lora_dropout                = LORA_DROPOUT,
    lora_target_modules         = ["q", "v"],
    reward_max_length           = 128,
)
config.use_4bit = True

rl_trainer = CEFRRLTrainer(config)

# resume logic
start_epoch, start_step = 0, 0
if RESUME:
    meta = find_latest_checkpoint(OUTPUT_DIR, GDRIVE_DIR)
    if meta:
        load_checkpoint_into_trainer(rl_trainer, meta["path"])
        start_epoch = meta["epoch"]
        start_step  = meta["step"]
        print(f"Resuming from epoch {start_epoch}, step {start_step}")
    else:
        print("No checkpoint found — starting from scratch")

# pre-training generation check
print("\n=== PRE-TRAINING GENERATION CHECK ===")
pre_outputs = rl_trainer.generation_debug(num_examples=5)

# build prompt batches
from data_utils import read_complicated_lines
training_sentences = read_complicated_lines(TRAIN_DATA_PATH)
if MAX_SENTENCES:
    training_sentences = training_sentences[:MAX_SENTENCES]
print(f"Training on {len(training_sentences)} sentences")

template          = rl_trainer._get_template()
generation_kwargs = rl_trainer._get_generation_kwargs(debug=False)

all_prompts = []
for i in range(0, len(training_sentences), BATCH_SIZE):
    batch = [
        rl_trainer._build_prompt_text(template.format(s))
        for s in training_sentences[i:i + BATCH_SIZE]
    ]
    all_prompts.append(batch)

# train
run_training(
    rl_trainer, config, all_prompts,
    start_epoch, start_step, generation_kwargs,
    output_dir       = OUTPUT_DIR,
    gdrive_dir       = GDRIVE_DIR,
    checkpoint_every = CHECKPOINT_EVERY,
)

# final checkpoint
final_local = os.path.join(OUTPUT_DIR, "final_checkpoint")
rl_trainer.save_checkpoint(final_local)
try:
    final_drive = os.path.join(GDRIVE_DIR, "final_checkpoint")
    if os.path.exists(final_drive):
        shutil.rmtree(final_drive)
    shutil.copytree(final_local, final_drive)
    print(f"Final checkpoint on Drive: {final_drive}")
except Exception as e:
    print(f"[WARNING] Final Drive copy failed: {e}")

# post-training generation check
print("\n=== POST-TRAINING GENERATION CHECK ===")
post_outputs = rl_trainer.generation_debug(num_examples=5)

print("\n=== BEFORE vs AFTER ===")
for pre, post in zip(pre_outputs, post_outputs):
    print(f"\nSOURCE : {pre['source']}")
    print(f"BEFORE : {pre['generated']}")
    print(f"AFTER  : {post['generated']}")

# save results to Drive
import json as json_mod
results = []
for pre, post in zip(pre_outputs, post_outputs):
    results.append({
        "source" : pre["source"],
        "before" : pre["generated"],
        "after"  : post["generated"],
        "level"  : CEFR_LEVEL,
    })
out_path = f"{GDRIVE_DIR}/results_level_{CEFR_LEVEL}.json"
with open(out_path, "w", encoding="utf-8") as f:
    json_mod.dump(results, f, ensure_ascii=False, indent=2)
print(f"Results saved to {out_path}")

In [ ]:
#  CELL 4 — Smoke test + Mini debug

print("SMOKE TEST")
smoke = rl_trainer.smoke_test_one_step()

if smoke is not None:
    print("\nMINI DEBUG (10 steps)")
    mini_results = rl_trainer.mini_train_debug(num_examples=10)

    kls = [r["kl"] for r in mini_results if "kl" in r]
    if kls:
        print(f"\nKL range: {min(kls):.4f} -> {max(kls):.4f}")
        if min(kls) < 0:
            print("WARNING: Negative KL — do NOT proceed to full training")
        else:
            print("KL stable — safe to run Cell 3")
else:
    print("Smoke test failed — check model and config before running Cell 3")